# 03 · SPAM spatial weights

Reproduce the original 5-arcminute SPAM → 0.5° LPJmL weighting logic. Uses **irrigated harvested area**, not physical irrigated area. Requires the five source parquets and frozen harmonized SPAM totals.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'

FABLE_TO_LPJML = {
 'cassava':'cas','corn':'mai','millet':'mil','groundnut':'nut',
 'peas':'pea','rapeseed':'rap','rice':'ric','sugarbeet':'sgb',
 'soyabean':'soy','sugarcane':'sug','sunflower':'sun','wheat':'whe'}
YEARS = [2000,2005,2010,2015,2020]
SRC = DATA / 'spam'
OUT = RUNS / 'spam-weights'

In [ ]:
def column(df, choices):
    for name in choices:
        if name in df.columns:return name
    raise KeyError(f'None of {choices} in {list(df.columns)}')

def prepare_weights(harmonized, source_spatial, year):
    h = harmonized[harmonized['year'].eq(year)].copy()
    h['fable_product_norm'] = h['fable_product'].astype(str).str.strip().str.lower()
    h = h[h.fable_product_norm.isin(FABLE_TO_LPJML)]
    area_col = column(h,['harvested_irrigated_ha','corrected_irrigated_ha'])
    targets = h.groupby(['country_iso3','fable_product_norm'],as_index=False)[area_col].sum()
    targets = targets.rename(columns={area_col:'target_irrigated_ha'})
    s=source_spatial.copy()
    for target,choices in {'country_iso3':['country_iso3','iso3'],
       'fable_product_norm':['fable_product_norm','fable_product'],
       'lon':['lon','longitude'],'lat':['lat','latitude'],
       'source_irrigated_cell':['harvested_irrigated_ha','irrigated_harvested_ha'],
       'source_total_cell':['harvested_total_ha','total_harvested_ha']}.items():
        found=column(s,choices)
        if target!=found:s=s.rename(columns={found:target})
    s['fable_product_norm']=s.fable_product_norm.astype(str).str.strip().str.lower()
    s=s[s.fable_product_norm.isin(FABLE_TO_LPJML)].copy()
    keys=['country_iso3','fable_product_norm']
    if s[['lon','lat','source_irrigated_cell','source_total_cell']].isna().any().any():
        raise ValueError('Null source SPAM coordinates or area')
    if (s[['source_irrigated_cell','source_total_cell']]<0).any().any():
        raise ValueError('Negative source SPAM area')
    totals=s.groupby(keys,as_index=False).agg(source_irrigated_ha=('source_irrigated_cell','sum'),
                                               source_total_harvested_ha=('source_total_cell','sum'))
    plan=targets.merge(totals,on=keys,how='left')
    plan[['source_irrigated_ha','source_total_harvested_ha']]=plan[['source_irrigated_ha','source_total_harvested_ha']].fillna(0)
    plan['method']=np.select([plan.target_irrigated_ha<=0,
       (plan.target_irrigated_ha>0)&(plan.source_irrigated_ha>0),
       (plan.target_irrigated_ha>0)&(plan.source_irrigated_ha<=0)&(plan.source_total_harvested_ha>0)],
       ['ZERO_IRRIGATED_TARGET','IRRIGATED_PATTERN_SCALED_TO_CALCULATOR',
        'TOTAL_HARVESTED_PATTERN_SAME_YEAR'],default='UNRESOLVED_POSITIVE_TARGET')
    if plan.method.eq('UNRESOLVED_POSITIVE_TARGET').any():
        raise ValueError('Positive calculator targets lack SPAM spatial support; no imputation allowed')
    work=s.merge(plan,on=keys,how='inner',validate='many_to_one')
    good=work.method.eq('IRRIGATED_PATTERN_SCALED_TO_CALCULATOR')
    fallback=work.method.eq('TOTAL_HARVESTED_PATTERN_SAME_YEAR')
    work['corrected_irrigated_ha']=0.0
    work.loc[good,'corrected_irrigated_ha']=(work.loc[good,'source_irrigated_cell']*
        work.loc[good,'target_irrigated_ha']/work.loc[good,'source_irrigated_ha'])
    work.loc[fallback,'corrected_irrigated_ha']=(work.loc[fallback,'source_total_cell']*
        work.loc[fallback,'target_irrigated_ha']/work.loc[fallback,'source_total_harvested_ha'])
    check=work.groupby(keys)['corrected_irrigated_ha'].sum().reset_index()
    chk=plan.merge(check,on=keys,how='left').fillna({'corrected_irrigated_ha':0.0})
    if not np.allclose(chk.corrected_irrigated_ha,chk.target_irrigated_ha,rtol=1e-7,atol=1e-6):
        raise ValueError('SPAM spatial allocation fails country-product area conservation')
    lon=np.floor((work.lon.astype(float).to_numpy()+180)/.5).astype(int).clip(0,719)
    lat=np.floor((work.lat.astype(float).to_numpy()+90)/.5).astype(int).clip(0,359)
    work['lpjml_lon']=-179.75+0.5*lon
    work['lpjml_lat']=-89.75+0.5*lat
    work['lpjml_crop_code']=work.fable_product_norm.map(FABLE_TO_LPJML)
    work['year']=int(year)
    result=work.groupby(['country_iso3','year','fable_product_norm','lpjml_crop_code',
                         'lpjml_lon','lpjml_lat'],as_index=False)['corrected_irrigated_ha'].sum()
    return result,plan

In [ ]:
HARM = SRC/'05_FABLE_C_SPAM_historical_input_2000_2020_HARMONIZED.csv.gz'
if not HARM.is_file(): HARM=HARM.with_suffix('')
if not HARM.is_file():raise FileNotFoundError(HARM)
harmonized=pd.read_csv(HARM,low_memory=False)
weights=[];allocations=[]
for year in YEARS:
    source=SRC/'spatial'/f'{year}.parquet'
    if not source.is_file():raise FileNotFoundError(source)
    grid,plan=prepare_weights(harmonized,pd.read_parquet(source),year)
    weights.append(grid);allocations.append(plan.assign(year=year))
all_weights=pd.concat(weights,ignore_index=True)
allocation_qc=pd.concat(allocations,ignore_index=True)
print(all_weights.groupby('year').agg(rows=('country_iso3','size'),
                                       corrected_area_ha=('corrected_irrigated_ha','sum')))
print(allocation_qc.groupby(['year','method']).size())

In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
files={
 '00_FABLE_SPAM_corrected_irrigated_area_05deg_2000_2005.parquet':[2000,2005],
 '00_FABLE_SPAM_corrected_irrigated_area_05deg_2010_2015_2020.parquet':[2010,2015,2020],
}
if any((OUT/f).exists() for f in [*files,'spatial-allocation-qc.csv']):
    raise FileExistsError('Output exists; use a fresh runs directory')
for filename,years in files.items():
    all_weights[all_weights.year.isin(years)].to_parquet(OUT/filename,index=False)
allocation_qc.to_csv(OUT/'spatial-allocation-qc.csv',index=False)
print('SPAM weights written:',OUT)